# تدريب مصنّف هجمات عربي لـ درع (Dir')
**قبل البدء:** من القائمة اختاري `Runtime ← Change runtime type ← T4 GPU`. ثم شغّلي الخلايا بالترتيب (Shift+Enter).
المدة التقريبية: 10 إلى 30 دقيقة. **ملاحظة:** هذا الدفتر لم يُجرَّب في بيئة التطوير (لا يوجد GPU ولا اتصال بـ Hugging Face هناك)، فإذا ظهر خطأ أرسليه لي كما هو.

In [ ]:
!pip -q install transformers datasets accelerate

ارفعي ملف `dir_shield_v2.zip` (الحزمة اللي أرسلتها لك) عند ظهور زر الرفع:

In [ ]:
from google.colab import files
up = files.upload()
!unzip -qo dir_shield_v2.zip
%cd dir_shield
!python train_data.py

## التدريب
نضبط نموذج `xlm-roberta-base` (متعدد اللغات يشمل العربية) ليصنّف: عادي = 0، هجوم = 1.

In [ ]:
import json, numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding)
from normalize import normalize

MODEL = "xlm-roberta-base"
def load(p): return [json.loads(l) for l in open(p, encoding="utf-8")]
tr, va = load("train.jsonl"), load("val.jsonl")
tok = AutoTokenizer.from_pretrained(MODEL)

def enc(rows):
    ds = Dataset.from_dict({"text": [normalize(r["text"]).text for r in rows], "label": [r["label"] for r in rows]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)

dtr, dva = enc(tr), enc(va)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(output_dir="out", num_train_epochs=4, per_device_train_batch_size=16, learning_rate=2e-5,
                         save_strategy="no", fp16=torch.cuda.is_available(), report_to="none", seed=7, logging_steps=20)
trainer = Trainer(model=model, args=args, train_dataset=dtr, data_collator=DataCollatorWithPadding(tok))
trainer.train()

pred = trainer.predict(dva)
yhat = pred.predictions.argmax(-1); y = np.array(dva["label"])
print("val accuracy:", round(float((yhat == y).mean()), 3), "| attacks caught:", int(((yhat == 1) & (y == 1)).sum()), "/", int((y == 1).sum()),
      "| false alarms:", int(((yhat == 1) & (y == 0)).sum()), "/", int((y == 0).sum()))
print("تنبيه: val من نفس القوالب، فرقمها متفائل. الاختبار الحقيقي هو الخلية التالية.")

## الاختبار الحقيقي على جمل كتبتها يدوياً (لم يرها النموذج)

In [ ]:
model.save_pretrained("dir_attack_clf"); tok.save_pretrained("dir_attack_clf")
import eval_data as ed, attacks
from hybrid import scan_hybrid
from ml_layer import MLLayer, MLConfig
from transformers import pipeline
clf = pipeline("text-classification", model="dir_attack_clf", top_k=None, truncation=True, max_length=128,
               device=0 if torch.cuda.is_available() else -1)

def p_attack(t):
    r = clf(normalize(t).text); r = r[0] if isinstance(r[0], list) else r
    return next(x["score"] for x in r if x["label"] == "LABEL_1")

A, B = ed.TEST_ATTACKS, ed.TEST_BENIGN
pa = [p_attack(a["text"]) for a in A]; pb = [p_attack(t) for t in B]
print(f"TEST: {len(A)} هجوماً، {len(B)} نصاً عادياً")
print("المصنّف وحده حسب العتبة:")
for th in (0.3, 0.5, 0.7, 0.9):
    print(f"  عتبة {th}: كشف {sum(p >= th for p in pa)}/{len(A)} | إنذارات كاذبة {sum(p >= th for p in pb)}/{len(B)}")

ml = MLLayer(MLConfig(use_knn=False, use_zero_shot=False, use_ner=False, clf_path="dir_attack_clf")); ml.load_all()
rules = (sum(attacks.scan(a["text"], a["src"]).verdict != "allow" for a in A), sum(attacks.scan(t).verdict != "allow" for t in B))
hyb = (sum(scan_hybrid(a["text"], a["src"], ml).verdict != "allow" for a in A), sum(scan_hybrid(t, "user", ml).verdict != "allow" for t in B))
print(f"\nقواعد فقط: كشف {rules[0]}/{len(A)} | إنذارات {rules[1]}/{len(B)}")
print(f"هجين (قواعد + مصنّف): كشف {hyb[0]}/{len(A)} | إنذارات {hyb[1]}/{len(B)}")
print("\nالأخطاء المهمة، هجمات فاتت المصنّف:")
for a, p in zip(A, pa):
    if p < 0.5: print("  -", round(p, 2), a["text"][:90].replace("\n", " "))

## الحفظ والتنزيل
نزّلي الملف، فكّي الضغط، وضعي المجلد `dir_attack_clf` بجانب `app.py` على جهازك. بعدها `python bench_ml.py`.

In [ ]:
!zip -qr dir_attack_clf.zip dir_attack_clf
files.download("dir_attack_clf.zip")